# Machine Learning
### Categorical naïve Bayes

**Categorical naïve Bayes** (**categorical NB**) is a probabilistic **classifier** that assumes 
- each feature is categorical (discrete), 
- and that the features are conditionally independent given the class.

It is a generalization of Bernoulli Naive Bayes — while Bernoulli NB deals with binary features, Categorical NB deals with features that can take on multiple discrete values (like color={“red”, “blue”, “green”}, size={“small”, “medium”, “large”}).



Thus, categorical NB assumes:
- $\boldsymbol{x}_i=[x_{i,1},x_{i,2},...,x_{i,q}]^T$ is a feature vector of length $q$, where each feature $x_{i,j}\in \mathcal{C}_j$ (categorical values)
  - In fact, each $\mathcal{C}_j$ is a finite set holding possible values for feature $j$.  
- $y_i$ is assumed to be class label for feature vector $\boldsymbol{x}_i$
  - Suppose we have $K$ classes such that $y_i\in \{1, 2,...,K\}$
- It uses the Naïve Bayes assumption that features are **conditionally independent** given the class. Thus:
  - $P(x_1,x_2,...,x_q|C=k)=\prod_{j=1}^{q}P(x_j|C=k)$ 

<hr>

Categorical NB is composed of two algorithms or phases: **Training** and **prediction**:
- **Training algorithm**: Given feature matrix $X$ (each row holds features of one sample) and corresponding class labels $\boldsymbol{y}$, we compute prior probabilities $P(C=k)$:
  - $P(C=k)=\frac{\text{number of samples with } C=k}{\text{total number of samples}}$
- We also compute the **conditional probabilities**: For feature $j$ and each possible value $c_j\in\mathcal{C}_j$ for this feature, given the class $k$, we compute:
  - $P(x_j=c_j|C=k)=\frac{count(x_j=c_j,C=k)+1}{\text{number of samples with } C=k +|\mathcal{C}_j|}$
    - Where $count(x_j=c_j,C=k)$ is the number of samples in class $k$ that their feature $j$ has value $c_j$. Also, $|\mathcal{C}_j|$ is the number of possible values for feature $j$.
    - We have used value of $1$ in the numerator and value $|c_j|$ in the denominator to prevent possible zero probabilities and to smooth the estimates. This is called **add-one smoothing** (or Laplace smoothing).
- **Prediction algorithm**: For the new 
feature vector $\boldsymbol{x}_{new}$, we choose the class with maximum **posterior**:
  - $C_{predicted}=arg max_k P(C=k|\boldsymbol{x}_{new})$
  - where $P(C=k|\boldsymbol{x}_{new})\propto P(\boldsymbol{x}_{new}|C=k)\cdot P(C=k)$
  - such that $P(\boldsymbol{x}_{new}|C=k)=\prod_{j=1}^q P(x_j=c_j|C=k)$ 

**Hint:** In the prediction algorithm, we may use **log probabilities** to avoid *underflow* and for *numerical stability*:
- $C_{predicted}=arg max_k score(k)$
  - $score(k)=log P(C=k)+\sum_{j=1}^q log P(x_j=c_j|C=k)$
<hr>

**Hint:** Some points on categorical NB:
- Simple, fast, interpretable
- Works well with discrete features
- No need for complex preprocessing
- Handles zero-frequency features with smoothing

<hr>

**Reminder:** We have computed here the class posterior probability by Bayes\' theorem:
<br> $P(C=k|\boldsymbol{x})=\frac{P(\boldsymbol{x}|C=k) \cdot P(C=k)}{P(\boldsymbol{x})}$
<br> Since the denominator $P(\boldsymbol{x})$ is constant for all classes, we ignore it for classification, and only compare the numerators in the prediction algorithm.
<hr>


In the following:
- We implement **categorial NB** from scratch in Python. 
- Afterwards, we choose the well-known **weather dataset** for experimenting with the NB. 
- Finally, our categorical NB is trained with a part of the dataset. Then, it is tested with the remaing unseen dataset.

<hr>

https://github.com/ostad-ai/Machine-Learning
<br> Explanation: https://www.pinterest.com/HamedShahHosseini/Machine-Learning/

In [1]:
# Import required modules
import math
from collections import defaultdict

In [4]:
class CategoricalNaiveBayes:
    def __init__(self):
        self.priors = {}       # P(class)
        self.conditionals = {} # P(feature|class)
        self.feature_values = defaultdict(set)  # All unique values per feature

    def fit(self, X, y):
        """
        X: list of lists, where each row is [feature1, feature2, ...]
        y: list of class labels (strings)
        """
        self.n_samples = len(X)
        n_features = len(X[0])
        classes = set(y)
        
        # Step 1: Get unique feature values for each column
        for i in range(n_features):
            for sample in X:
                self.feature_values[i].add(sample[i])

        # Step 2: Compute class counts and priors
        # We calculate the raw counts here so we can reuse them in Step 4
        class_counts = {cls: 0 for cls in classes}
        for label in y:
            class_counts[label] += 1
            
        # Calculate priors by dividing counts by total samples
        self.priors = {cls: count / self.n_samples for cls, count in class_counts.items()}

        # Step 3: Initialize conditionals and count feature occurrences per class
        for cls in classes:
            self.conditionals[cls] = defaultdict(lambda: defaultdict(int))
            
        for idx, sample in enumerate(X):
            cls = y[idx]
            for i, feature_val in enumerate(sample):
                self.conditionals[cls][i][feature_val] += 1

        # Step 4: Apply Laplace smoothing and normalize (OPTIMIZED)
        for cls in classes:
            n_c = class_counts[cls]  # <--- Use the pre-calculated total samples (N_c)
            
            for feature_idx in range(n_features):
                values = self.feature_values[feature_idx]
                denominator = n_c + len(values)  # Denominator is N_c + |V|
                
                for val in values:
                    count = self.conditionals[cls][feature_idx][val]
                    # Add 1 for Laplace smoothing, then normalize
                    self.conditionals[cls][feature_idx][val] = (count + 1) / denominator

        print("✅ Model trained.")
        
    def predict(self, X):
        """
        X: list of lists, where each row is [feature1, feature2, ...]
        Returns: list of predicted class labels
        """
        predictions = []
        for sample in X:
            log_scores = {}
            for cls in self.priors:
                log_scores[cls] = math.log(self.priors[cls])
                
                for i, feature_val in enumerate(sample):
                    # Use 1e-9 for completely unseen values to prevent math.log(0) error
                    prob = self.conditionals[cls][i].get(feature_val, 1e-9)
                    log_scores[cls] += math.log(prob)
                    
            # Pick class with highest score
            predicted_cls = max(log_scores, key=log_scores.get)
            predictions.append(predicted_cls)
            
        return predictions

    def evaluate(self, X, y):
        """
        X: test data (list of lists)
        y: true labels
        Returns: accuracy
        """
        predictions = self.predict(X)
        correct = sum(1 for y_pred, y_true in zip(predictions, y) if y_pred == y_true)
        return correct / len(y)

In [5]:
# --- Example with Weather Dataset ---
# Weather dataset: 14 samples, 4 features, 1 target
weather_data = [
    ['Sunny', 'Hot', 'High', 'Weak', 'No'],
    ['Sunny', 'Hot', 'High', 'Strong', 'No'],
    ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Sunny', 'Cool', 'Normal', 'Strong', 'No'],
    ['Rain', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Overcast', 'Mild', 'High', 'Weak', 'Yes'],
    ['Overcast', 'Hot', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Strong', 'No'],
    ['Sunny', 'Mild', 'High', 'Strong', 'No']
]

# Separate features (X) and target (y)
X = [row[:4] for row in weather_data]  # first 4 columns are features
y = [row[4] for row in weather_data]   # last column is target

# But since we have 14 samples, let's use 10 for training and 4 for testing (we'll do it manually)

train_X = X[:10]
train_y = y[:10]
test_X = X[10:]
test_y = y[10:]

# Initialize and train
nb = CategoricalNaiveBayes()

# Train
nb.fit(train_X, train_y)

# Predict
predictions = nb.predict(test_X)

# Evaluate
accuracy = nb.evaluate(test_X, test_y)
print('-'*30)
print('Trained model is tested with unseen data:')
print(f"✅ Accuracy: {accuracy:.3f}")

# Print predictions and true values
print('--- Details of testing ---')
print("Predicted labels vs. True labels")
for y_pred, y_true in zip(predictions, test_y):
    print(f"Predicted label: {y_pred};  True label: {y_true}")

# 🧪 Optional: Try predicting a new sample
new_sample = ['Sunny', 'Mild', 'High', 'Strong']
print('-'*30)
print(f"Predicting for new sample: {new_sample}:")
pred = nb.predict([new_sample])[0]
print(f"Predicted label: {pred}")

✅ Model trained.
------------------------------
Trained model is tested with unseen data:
✅ Accuracy: 0.750
--- Details of testing ---
Predicted labels vs. True labels
Predicted label: Yes;  True label: Yes
Predicted label: Yes;  True label: Yes
Predicted label: Yes;  True label: No
Predicted label: No;  True label: No
------------------------------
Predicting for new sample: ['Sunny', 'Mild', 'High', 'Strong']:
Predicted label: No
